In [2]:
import pandas as pd
import numpy as np
print(pd.__version__, np.__version__)

3.0.6 2.4.6


In [34]:
df = pd.read_csv("../data/auth_logs.csv", parse_dates=["time"])
df.head()

,time,ip,user,event,status
0,2026-10-01 08:00:08,10.0.0.24,dave,login,success
1,2026-10-01 08:00:11,10.0.0.26,dave,login,success
2,2026-10-01 08:00:19,10.0.0.16,bob,login,success
3,2026-10-01 08:00:30,10.0.0.29,dave,login,success
4,2026-10-01 08:00:33,10.0.0.16,bob,login,success


In [35]:
df.shape, df.dtypes

((3200, 5),
 time      datetime64[us]
 ip                   str
 user                 str
 event                str
 status               str
 dtype: object)

In [36]:
df["status"].value_counts()

status
success    2849
failed      351
Name: count, dtype: int64

In [37]:
failed = df[df["status"] == "failed"]
failed["ip"].value_counts().head()

ip
203.0.113.50    200
10.0.0.22        12
10.0.0.21        11
10.0.0.24        10
10.0.0.17         9
Name: count, dtype: int64

In [38]:
atk = df[df["ip"] == "203.0.113.50"]
print(atk["time"].min(), atk["time"].max())
print(atk["user"].nunique(), len(atk))

2026-10-01 11:00:00 2026-10-01 11:04:59
5 200


In [39]:
failed.groupby("ip").size().sort_values(ascending=False).head(20)
failed.groupby("user").size()

user
alice    78
bob      58
carol    77
dave     64
erin     74
dtype: int64

In [40]:
w = (failed
     .groupby([pd.Grouper(key="time", freq="5min"), "ip"])
     .agg(n_failed=("status", "size"), n_users=("user", "nunique"))
     .reset_index())
w.sort_values("n_failed", ascending=False).head()

,time,ip,n_failed,n_users
67,2026-10-01 11:00:00,203.0.113.50,200,5
31,2026-10-01 09:40:00,10.0.0.11,2,2
79,2026-10-01 11:30:00,10.0.0.17,2,2
109,2026-10-01 12:40:00,10.0.0.21,2,2
112,2026-10-01 12:45:00,10.0.0.28,2,2


In [41]:
w["is_attack"] = (w["ip"] == "203.0.113.50").astype(int)
w["rule_flag"] = (w["n_failed"] > 5).astype(int)
pd.crosstab(w["is_attack"], w["rule_flag"])

rule_flag,0,1
is_attack,,
0,146,0
1,0,1


In [42]:
w.groupby("ip")["n_failed"].agg(["count", "sum", "max"]).sort_values("sum", ascending=False).head(5)

,count,sum,max
ip,,,
203.0.113.50,1,200,200
10.0.0.22,12,12,1
10.0.0.21,10,11,2
10.0.0.24,10,10,1
10.0.0.17,8,9,2
